# 03 · 선행연구 원문에서 '상장폐지·생존편의' 문장 찾기

| 항목 | 내용 |
|---|---|
| 하는 일 | `논문/상장 vs 상장폐지/`의 PDF 7편에서 확인 목록의 말이 PDF 몇 쪽에 나오는지 찾아 출력 (A9 근거 점검용) |
| 읽는 파일 | `논문/상장 vs 상장폐지/*.pdf` (PyMuPDF 필요) |
| 만드는 파일 | 없음 (화면 출력만. 뽑은 글자는 저장하지 않음 — 논문 저작권) |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 10초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

**왜 만들었나** — 논문 제목이나 요약만 보고 "이 논문이 생존편의를 다뤘다"고 쓰면 안 되기 때문에(프로젝트 지침 5번), 실제 본문에 그 말이 있는지 PDF에서 직접 찾아 쪽수를 확인합니다. 아래 `CHECKS`가 논문별 확인 목록입니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""국내 선행연구 PDF에서 '상장폐지·생존편의' 관련 문장이 어느 쪽에 있는지 확인한다. (A9 근거 점검용)

하는 일: Corp/논문/상장 vs 상장폐지/ 안의 PDF에서 글자를 뽑아, 아래 확인 목록의 말이 몇 쪽에 나오는지 출력한다.
읽기만 한다. 아무 파일도 만들지 않는다. 뽑은 글자는 저장하지 않는다(논문 저작권).

실행: python -X utf8 논문_원문_확인.py
필요: pip install pymupdf

주의
- 쪽수는 'PDF 파일 안의 쪽'이다. 학술지 쪽수와 다를 수 있다(예: 김성환·김미나 PDF 6쪽 = 학술지 1258쪽).
- 일부 PDF는 한글 띄어쓰기가 사라진 채 추출된다(예: 이석근 외 2016). 그래서 확인 말은 공백을 지운 글자에서 찾는다.
- '찾지 못함'은 추출된 글자에서 못 찾았다는 뜻이다. 표나 그림 속 글자는 확인하지 못한다.
"""
import re
import sys
from pathlib import Path
import fitz  # PyMuPDF
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

try:
    HERE = Path(__file__).resolve().parent      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
PAPERS = HERE.parents[1] / "논문" / "상장 vs 상장폐지"
# (PDF 파일 이름의 일부, 논문 설명, 확인할 말 목록). 확인 말은 공백을 지운 글자에서 찾는다.
CHECKS = [
    ("감사인의 평판", "김성환·김미나(2014) 대한경영학회지 27(8)",
     ["상장폐지기업도포함", "불균형패널", "재무자료를구할수없는기업은제외", "생존자편의를제거"]),
    ("한국증권시장에서의 장기성과", "정형찬(2007) Asia-Pacific J. of Financial Studies 36(2)",
     ["생존편의(survivorbias)를최소화", "Delisted", "하향편의"]),
    ("상장폐지기업의 상장폐지전", "이석근·조광희·양동훈(2016) 회계저널 25(2)",
     ["28,224", "자본완전잠식", "회계정보미비", "폐업으로인하여"]),
    ("상장폐지기업 예측모형", "배기수·정설희(2011) 세무회계연구 30",
     ["상장공시시스템", "Kis－ValueⅢ", "81개"]),
    ("최고경영자의 특성", "장지영·김지령(2015) 회계ㆍ세무와 감사 연구 57(1)",
     ["상장폐지및관리대상", "생존편의문제가제기"]),
    ("통화정책과_기업_재무", "조상섭·배기수(2026) 상업경영연구 40(2)",
     ["TS2000", "균형패널", "이자비용을매출액", "상장폐지", "생존"]),
    ("미 연준의 긴축", "도영웅(2026) 국제경제연구 32(3)",
     ["Value Search", "하위5%", "상장폐지", "생존"]),
]

#### `squeeze()` — 공백을 모두 지운 글자로 바꿈(일부 PDF는 한글 띄어쓰기가 사라진 채 추출되어서)

In [3]:
def squeeze(text):
    return re.sub(r"\s+", "", text)

#### `main()` — 논문마다 PDF를 열어 확인 말이 몇 쪽에 있는지 출력

In [4]:
def main():
    if not PAPERS.exists():
        sys.exit(f"논문 폴더가 없습니다: {PAPERS}")
    for key, label, phrases in CHECKS:
        hit = [p for p in PAPERS.glob("*.pdf") if key in p.name]
        print("=" * 70)
        print(label)
        if not hit:
            print("  PDF를 찾지 못함(폴더에 있는지, 이름에 아래 말이 들어 있는지 확인):", key)
            continue
        doc = fitz.open(hit[0])
        pages = [squeeze(pg.get_text()) for pg in doc]
        print(f"  파일: {hit[0].name} | PDF {len(doc)}쪽")
        for ph in phrases:
            target = squeeze(ph)
            found = [i + 1 for i, t in enumerate(pages) if target in t]
            print(f"  - '{ph}': " + (f"PDF {found}쪽" if found else "찾지 못함"))

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [5]:
main()

김성환·김미나(2014) 대한경영학회지 27(8)
  파일: 감사인의 평판과 회계감사의견이 상장기업 가치에 미치는 영향.pdf.pdf | PDF 15쪽
  - '상장폐지기업도포함': PDF [4, 13]쪽
  - '불균형패널': PDF [6, 7]쪽
  - '재무자료를구할수없는기업은제외': PDF [6]쪽
  - '생존자편의를제거': PDF [1, 4, 6, 9, 11, 13]쪽
정형찬(2007) Asia-Pacific J. of Financial Studies 36(2)


  파일: 한국증권시장에서의 장기성과 측정모형의 검정력과 통계적 오류.pdf.pdf | PDF 44쪽
  - '생존편의(survivorbias)를최소화': PDF [7]쪽
  - 'Delisted': PDF [14]쪽
  - '하향편의': PDF [14, 15, 21, 22, 25, 28, 33]쪽
이석근·조광희·양동훈(2016) 회계저널 25(2)
  파일: 상장폐지기업의 상장폐지전 회계정보의 가치관련성.pdf | PDF 23쪽
  - '28,224': PDF [9]쪽
  - '자본완전잠식': PDF [9]쪽
  - '회계정보미비': PDF [9]쪽
  - '폐업으로인하여': PDF [11]쪽
배기수·정설희(2011) 세무회계연구 30


  파일: 상장폐지기업 예측모형에 관한 실증연구.pdf | PDF 16쪽
  - '상장공시시스템': PDF [5]쪽
  - 'Kis－ValueⅢ': PDF [5]쪽
  - '81개': PDF [1, 5, 6]쪽
장지영·김지령(2015) 회계ㆍ세무와 감사 연구 57(1)
  파일: 06 최고경영자의 특성이 투자효율성에 미치는 영향.pdf | PDF 32쪽
  - '상장폐지및관리대상': PDF [12]쪽
  - '생존편의문제가제기': PDF [12]쪽
조상섭·배기수(2026) 상업경영연구 40(2)


  파일: 통화정책과_기업_재무_조정에.pdf | PDF 22쪽
  - 'TS2000': PDF [11]쪽
  - '균형패널': PDF [11]쪽
  - '이자비용을매출액': PDF [11]쪽
  - '상장폐지': 찾지 못함
  - '생존': 찾지 못함
도영웅(2026) 국제경제연구 32(3)


  파일: 미 연준의 긴축 충격이 한국 기업의 주가, 자금조달, 투자에 미치는 영향.pdf | PDF 36쪽
  - 'Value Search': PDF [6, 7, 13, 18, 19]쪽
  - '하위5%': PDF [7, 8]쪽
  - '상장폐지': 찾지 못함
  - '생존': 찾지 못함


### 읽는 법과 주의

- 쪽수는 **PDF 파일 안의 쪽**입니다. 학술지에 인쇄된 쪽수와 다를 수 있습니다.
- "찾지 못함"은 *추출된 글자에서* 못 찾았다는 뜻입니다. 표·그림 속 글자는 확인하지 못하므로 "논문에 없다"는 뜻이 아닙니다.
- 이 출력은 문장이 **있는 위치**만 알려 줍니다. 그 문장이 우리 연구에서 무엇을 뒷받침하는지는 PDF 해당 쪽을 직접 읽고 판단해야 합니다.